# CS-4365 Colab Baseline Workflow

This notebook is a **single Colab workflow** that does the whole Checkpoint 2 baseline:

1. clones the GitHub repo
2. installs dependencies
3. mounts Google Drive
4. helps locate the Lending Club dataset in Drive
5. imports the project code from `src/`
6. loads and cleans the dataset
7. selects baseline features
8. creates the temporal split
9. trains the baseline logistic regression model
10. evaluates AUC and F1 by year
11. saves result files
12. optionally copies outputs to Google Drive

Run this notebook **top to bottom in Colab**.


## Step 0: Clone the repo and install dependencies


In [ ]:
import os

REPO_NAME = "CS-4365"
REPO_URL = "https://github.com/ndave24/CS-4365.git"

if not os.path.exists(f"/content/{REPO_NAME}"):
    !git clone {REPO_URL}
else:
    print("Repo already exists in /content, skipping clone.")

%cd /content/CS-4365
!pip install -r requirements.txt


## Step 1: Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## Step 2: Set up repo imports


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("/content/CS-4365")

if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

print("Repo root:", REPO_ROOT)
print("Repo exists:", REPO_ROOT.exists())


## Step 3: Find the dataset in Google Drive

This searches Google Drive for files beginning with `lending_club`.


In [ ]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")
matches = list(drive_root.rglob("lending_club*"))

if len(matches) == 0:
    print("No files found matching 'lending_club*'.")
    print("Upload the dataset to Google Drive first, then rerun this cell.")
else:
    print("Found possible dataset files:")
    for m in matches[:100]:
        print(m)


## Step 4: Set the dataset path

Copy one of the paths printed above and paste it into `DATA_PATH`.


In [ ]:
from pathlib import Path

# TODO: replace this with the real path you found above.
DATA_PATH = Path("/content/drive/MyDrive/datasets/lending_club.parquet")

print("DATA_PATH:", DATA_PATH)
print("Exists? ->", DATA_PATH.exists())


## Step 5: Optional CSV to Parquet conversion

If your dataset is only a CSV, run this once to convert it to Parquet.
Leave `RUN_CONVERSION = False` if you already have a Parquet file.


In [ ]:
RUN_CONVERSION = False

if RUN_CONVERSION:
    import pandas as pd

    if DATA_PATH.suffix.lower() != ".csv":
        raise ValueError("Set DATA_PATH to the CSV file before running conversion.")

    parquet_path = DATA_PATH.with_suffix(".parquet")
    print("Reading CSV from:", DATA_PATH)
    df_csv = pd.read_csv(DATA_PATH, low_memory=False)
    df_csv.to_parquet(parquet_path, index=False)
    print("Saved Parquet to:", parquet_path)
else:
    print("Skipping conversion.")


## Step 6: Import project modules


In [ ]:
from src.load_data import load_and_clean_dataset, summarize_dataset
from src.preprocess import (
    PreprocessConfig,
    get_model_feature_groups,
    summarize_feature_groups,
)
from src.temporal_split import (
    TemporalSplitConfig,
    make_temporal_split,
    describe_split,
    describe_test_years,
)
from src.models.logistic import fit_logistic_pipeline
from src.evaluate import (
    add_time_gap_column,
    evaluate_temporal_by_year,
    summarize_temporal_results,
    plot_temporal_metrics,
    save_temporal_metrics,
)


## Step 7: Load and clean the dataset


In [ ]:
df = load_and_clean_dataset(DATA_PATH)
summarize_dataset(df)


## Step 8: Inspect columns and a few rows


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nHead:")
display(df.head())


## Step 9: Check target distribution and year distribution


In [ ]:
print("Target distribution:")
display(df["Default"].value_counts(dropna=False))

print("\nDefault rate:")
print(df["Default"].mean())

print("\nYear distribution:")
display(df["year"].value_counts().sort_index())


## Step 10: Configure baseline preprocessing

For the Checkpoint 2 baseline:
- no text
- no ID
- drop zip code
- keep categorical columns with manageable cardinality


In [ ]:
preprocess_config = PreprocessConfig(
    include_text=False,
    include_id=False,
    drop_zip_code=True,
    max_categorical_cardinality=50,
)


## Step 11: Build feature groups


In [ ]:
feature_groups = get_model_feature_groups(df, config=preprocess_config)
summarize_feature_groups(feature_groups)


## Step 12: Inspect selected features


In [ ]:
print("Feature columns:")
print(feature_groups["feature_cols"])

print("\nNumeric columns:")
print(feature_groups["numeric_cols"])

print("\nCategorical columns:")
print(feature_groups["categorical_cols"])

print("\nText columns:")
print(feature_groups["text_cols"])


## Step 13: Configure the temporal split

The baseline uses:
- train on years up to 2014
- test on 2015, 2016, 2017, 2018


In [ ]:
TRAIN_END_YEAR = 2014
TEST_YEARS = (2015, 2016, 2017, 2018)

split_config = TemporalSplitConfig(
    train_end_year=TRAIN_END_YEAR,
    test_years=TEST_YEARS,
)

split_dict = make_temporal_split(df, config=split_config)


## Step 14: Summarize the split


In [ ]:
split_summary_df = describe_split(split_dict)
split_summary_df


## Step 15: Summarize test years


In [ ]:
test_year_summary_df = describe_test_years(split_dict["test_df"])
test_year_summary_df


## Step 16: Fit the baseline logistic regression


In [ ]:
model = fit_logistic_pipeline(
    train_df=split_dict["train_df"],
    feature_groups=feature_groups,
)

print("Model fit complete.")


## Step 17: Evaluate temporal performance by year


In [ ]:
results_df = evaluate_temporal_by_year(
    model=model,
    test_df=split_dict["test_df"],
    feature_groups=feature_groups,
)

results_df = add_time_gap_column(results_df, train_end_year=TRAIN_END_YEAR)
results_df


## Step 18: Compact summary table


In [ ]:
summarize_temporal_results(results_df)


## Step 19: Plot AUC and F1 by test year


In [ ]:
plot_temporal_metrics(
    results_df=results_df,
    use_time_gap=False,
    title="Temporal Performance of Logistic Regression",
)


## Step 20: Plot AUC and F1 by time gap


In [ ]:
plot_temporal_metrics(
    results_df=results_df,
    use_time_gap=True,
    title="Temporal Performance vs Time Gap",
)


## Step 21: Save outputs to the repo `results/` folder


In [ ]:
results_dir = REPO_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

save_temporal_metrics(results_df, results_dir / "temporal_metrics.csv")
split_summary_df.to_csv(results_dir / "split_summary.csv", index=False)
test_year_summary_df.to_csv(results_dir / "test_year_summary.csv", index=False)

plot_temporal_metrics(
    results_df=results_df,
    output_path=results_dir / "temporal_auc_f1.png",
    use_time_gap=False,
    title="Temporal Performance of Logistic Regression",
)

plot_temporal_metrics(
    results_df=results_df,
    output_path=results_dir / "temporal_auc_f1_time_gap.png",
    use_time_gap=True,
    title="Temporal Performance vs Time Gap",
)

print("Saved outputs to:", results_dir)
for p in sorted(results_dir.iterdir()):
    print("-", p.name)


## Step 22: Optional copy of outputs to Google Drive


In [ ]:
COPY_RESULTS_TO_DRIVE = True
DRIVE_RESULTS_DIR = Path("/content/drive/MyDrive/CS4365_results")

if COPY_RESULTS_TO_DRIVE:
    DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    for p in results_dir.iterdir():
        if p.is_file():
            target = DRIVE_RESULTS_DIR / p.name
            target.write_bytes(p.read_bytes())

    print("Copied results to:", DRIVE_RESULTS_DIR)
    for p in sorted(DRIVE_RESULTS_DIR.iterdir()):
        print("-", p.name)
else:
    print("Skipping copy to Google Drive.")


## Step 23: Optional git status check


In [ ]:
!git status


## Notes

- If `DATA_PATH.exists()` prints `False`, the dataset path is wrong.
- If you only have a CSV, either use it directly or convert it once to Parquet.
- This notebook is intended to be the **single Colab workflow** for generating the current baseline results.
